# 04 — Fine-tune on YOUR OWN frames

Loads base weights and continues training at a **low learning rate** on your own cultivator-camera frames. Keeps both classes.

**On this dataset, 04 is not optional — it is the notebook that makes the model work.** LettuceMOTS is a uniformly healthy crop: it teaches the detector where plants are and what a healthy canopy looks like, but it contains no affected plants to learn `unhealthy` from. The `unhealthy` class only becomes real here, from frames that actually contain it.

**Your frames must be labeled in YOLO detection format**, two classes, `0 = healthy` and `1 = unhealthy` — the same order as `health.yaml`, or every label is silently inverted. Point `OWN_DATA_YAML` at that yaml. Use only real captured/annotated frames — no synthetic or AI-generated images.

Collecting them: shoot the affected patches of a real field at the same camera height and angle as the deployment, in the same light. Aim for a few hundred `unhealthy` instances at minimum, and do not let them all come from one plot or one afternoon — otherwise the model learns that plot's soil colour or that hour's light rather than the plants.

> **DO NOT RUN** until your frames + labels exist and the config below points at them. This notebook ships **unrun** on purpose.

In [ ]:
import os, sys
from pathlib import Path

# Locate repo root (holds croprow_disease/utils.py) so the package imports
# regardless of the cwd the notebook is launched from.
REPO_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "croprow_disease" / "utils.py").is_file()
)
sys.path.insert(0, str(REPO_ROOT))
from croprow_disease import utils as U
from croprow_disease.health import HealthParams

CW = REPO_ROOT / "croprow_disease"
DATA_DIR = CW / "data"
MODELS_DIR = CW / "models"
RUNS_DIR = CW / "runs"
RESULTS_MD = CW / "RESULTS.md"

# ===================== CONFIG (edit here only) =====================
# Base weights from 03_train. Falls back to models/best.pt.
BASE_WEIGHTS = str(MODELS_DIR / "best.pt")

# Path to YOUR 2-class data yaml (train/val over your own frames).
# Prefer the env var; else edit the default. Placeholder on purpose.
OWN_DATA_YAML = os.environ.get("OWN_DATA_YAML", r"D:\path\to\your_frames\own.yaml")

IMGSZ      = 640
EPOCHS     = 50
BATCH      = 16
LR0        = 0.001                 # low LR for fine-tuning (10x below 03)
LRF        = 0.01
OPTIMIZER  = "auto"
FREEZE     = 10                    # freeze backbone layers (0 to disable)
PATIENCE   = 20
WORKERS    = 8
DEVICE     = 0
SEED       = 42

RUN_NAME   = f"finetune_own_health_{IMGSZ}"
# ===================================================================
print("base weights :", BASE_WEIGHTS)
print("own data yaml:", OWN_DATA_YAML)

## Environment check

In [ ]:
# This notebook needs the training/inference stack (torch + ultralytics),
# NOT installed in the light 01/02 env. See croprow_disease/requirements-train.txt.
try:
    import torch
    from ultralytics import YOLO
    import ultralytics
    print("torch      :", torch.__version__)
    print("ultralytics:", ultralytics.__version__)
    print("CUDA avail :", torch.cuda.is_available(),
          "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"))
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        f"Missing training dependency: {e.name}. Install "
        "croprow_disease/requirements-train.txt into the croprow Python 3.11 venv "
        "before running this notebook."
    ) from e

## Guard: confirm inputs exist and the class order matches

Stops with a clear message rather than fabricating data. The class-order check matters more than it looks: a yaml with `names: [unhealthy, healthy]` trains happily and inverts every prediction.

In [ ]:
import yaml as _yaml

if not Path(BASE_WEIGHTS).is_file():
    raise FileNotFoundError(
        f"Base weights not found: {BASE_WEIGHTS}. Run 03_train first, or set "
        "BASE_WEIGHTS to your trained checkpoint.")
if not Path(OWN_DATA_YAML).is_file():
    raise FileNotFoundError(
        f"Own-frames data yaml not found: {OWN_DATA_YAML}. Provide your labeled "
        "frames (2 classes) and point OWN_DATA_YAML at their yaml.")

own_cfg = _yaml.safe_load(Path(OWN_DATA_YAML).read_text())
own_names = list(own_cfg.get("names", []))
print("own classes:", own_names)
if own_names != U.CLASS_NAMES:
    raise ValueError(
        f"Class order mismatch: your yaml has {own_names}, this module uses "
        f"{U.CLASS_NAMES}. Ultralytics matches classes BY INDEX, so training "
        "with these swapped inverts every prediction silently. Fix the yaml "
        "(and the label files if their ids follow the old order) before "
        "continuing.")
print("inputs OK")

## Fine-tune

In [ ]:
model = YOLO(BASE_WEIGHTS)
results = model.train(
    data=OWN_DATA_YAML,
    imgsz=IMGSZ,
    epochs=EPOCHS,
    batch=BATCH,
    lr0=LR0,
    lrf=LRF,
    optimizer=OPTIMIZER,
    freeze=FREEZE,
    patience=PATIENCE,
    workers=WORKERS,
    device=DEVICE,
    seed=SEED,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
)
save_dir = Path(model.trainer.save_dir)
best = save_dir / "weights" / "best.pt"
print("fine-tuned best:", best)

import shutil
if best.is_file():
    dst = MODELS_DIR / "best_finetuned.pt"
    shutil.copy2(best, dst)
    print("copied ->", dst)

Evaluate the fine-tuned model on your own val split in `05_evaluate` — reported separately from LettuceMOTS, and with `labels="annotated"` since these classes are real human labels rather than the colour rule's output.